## 1. Import Libraries

In [ ]:
# Import libraries

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

## 2. Data Loading

In [ ]:
df = pd.read_csv("flights_sample_3m.csv")
print(df.head())

      FL_DATE                AIRLINE                AIRLINE_DOT AIRLINE_CODE  \
0  2019-01-09  United Air Lines Inc.  United Air Lines Inc.: UA           UA   
1  2022-11-19   Delta Air Lines Inc.   Delta Air Lines Inc.: DL           DL   
2  2022-07-22  United Air Lines Inc.  United Air Lines Inc.: UA           UA   
3  2023-03-06   Delta Air Lines Inc.   Delta Air Lines Inc.: DL           DL   
4  2020-02-23       Spirit Air Lines       Spirit Air Lines: NK           NK   

   DOT_CODE  FL_NUMBER ORIGIN          ORIGIN_CITY DEST  \
0     19977       1562    FLL  Fort Lauderdale, FL  EWR   
1     19790       1149    MSP      Minneapolis, MN  SEA   
2     19977        459    DEN           Denver, CO  MSP   
3     19790       2295    MSP      Minneapolis, MN  SFO   
4     20416        407    MCO          Orlando, FL  DFW   

               DEST_CITY  ...  DIVERTED  CRS_ELAPSED_TIME  ELAPSED_TIME  \
0             Newark, NJ  ...       0.0             186.0         176.0   
1            S

In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 32 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   FL_DATE                  object 
 1   AIRLINE                  object 
 2   AIRLINE_DOT              object 
 3   AIRLINE_CODE             object 
 4   DOT_CODE                 int64  
 5   FL_NUMBER                int64  
 6   ORIGIN                   object 
 7   ORIGIN_CITY              object 
 8   DEST                     object 
 9   DEST_CITY                object 
 10  CRS_DEP_TIME             int64  
 11  DEP_TIME                 float64
 12  DEP_DELAY                float64
 13  TAXI_OUT                 float64
 14  WHEELS_OFF               float64
 15  WHEELS_ON                float64
 16  TAXI_IN                  float64
 17  CRS_ARR_TIME             int64  
 18  ARR_TIME                 float64
 19  ARR_DELAY                float64
 20  CANCELLED                float64
 21  CANCELLA

## 3. Data Preparation

In [ ]:
# Convert FL_DATE to datetime
df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])

# Create YEAR, MONTH, MONTH_NAME
df["YEAR"] = df["FL_DATE"].dt.year
df["MONTH"] = df["FL_DATE"].dt.month
df["MONTH_NAME"] = df["FL_DATE"].dt.strftime("%b")

# Create ROUTE
df["ROUTE"] = df["ORIGIN"] + " → " + df["DEST"]

# Create DELAYED flag
df["DELAYED"] = (df["ARR_DELAY"] > 15).astype(int)

# Create DELAY_CATEGORY
df["DELAY_CATEGORY"] = np.where(
    df["CANCELLED"] == 1,
    "Cancelled",
    np.where(
        df["ARR_DELAY"] <= 0,
        "On Time / Early",
        np.where(
            df["ARR_DELAY"] <= 15,
            "Minor Delay",
            np.where(
                df["ARR_DELAY"] <= 60,
                "Moderate Delay",
                "Major Delay"
            )
        )
    )
)

# Create FLIGHT_STATUS
df["FLIGHT_STATUS"] = np.where(
    df["CANCELLED"] == 1,
    "Cancelled",
    np.where(df["ARR_DELAY"] > 15,
             "Delayed",
             "On Time")
)

# Fill delay-cause columns with 0
delay_cols = [
    "DELAY_DUE_CARRIER",
    "DELAY_DUE_WEATHER",
    "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY",
    "DELAY_DUE_LATE_AIRCRAFT"
]

df[delay_cols] = df[delay_cols].fillna(0)

# Create MAJOR_AIRPORT flag
airport_counts = (df.groupby("ORIGIN").size().reset_index(name="AIRPORT_FLIGHTS"))

df = df.merge(airport_counts,on="ORIGIN",how="left")
df["MAJOR_AIRPORT"] = np.where(df["AIRPORT_FLIGHTS"] >= 10000,"Yes","No")

# Create MAJOR_ROUTE flag
route_counts = (df.groupby("ROUTE").size().reset_index(name="ROUTE_FLIGHTS"))

df = df.merge(route_counts,on="ROUTE",how="left")
df["MAJOR_ROUTE"] = np.where(df["ROUTE_FLIGHTS"] >= 1000,"Yes","No")

## 4. Data Validation

In [ ]:
# Check Delay Time variables
print(df["ARR_DELAY"].min())
print(df["ARR_DELAY"].max())

print(df["DEP_DELAY"].min())
print(df["DEP_DELAY"].max())

-96.0
2934.0
-90.0
2966.0


In [ ]:
# Check Distance variable
print(df["DISTANCE"].min()) # Distance should never be 0

29.0


In [ ]:
# Check Flight Time variables

for col in [
    "AIR_TIME", # Flight time should never be 0
    "ELAPSED_TIME",
    "CRS_ELAPSED_TIME"
]:
    print(col)
    print(df[col].min())

AIR_TIME
8.0
ELAPSED_TIME
15.0
CRS_ELAPSED_TIME
1.0


In [ ]:
# Check flights with impossible delays

print(df[(df["CANCELLED"] == 1) & (df["ARR_DELAY"].notna())]) # A cancelled flight should not have an arrival delay.

Empty DataFrame
Columns: [FL_DATE, AIRLINE, AIRLINE_DOT, AIRLINE_CODE, DOT_CODE, FL_NUMBER, ORIGIN, ORIGIN_CITY, DEST, DEST_CITY, CRS_DEP_TIME, DEP_TIME, DEP_DELAY, TAXI_OUT, WHEELS_OFF, WHEELS_ON, TAXI_IN, CRS_ARR_TIME, ARR_TIME, ARR_DELAY, CANCELLED, CANCELLATION_CODE, DIVERTED, CRS_ELAPSED_TIME, ELAPSED_TIME, AIR_TIME, DISTANCE, DELAY_DUE_CARRIER, DELAY_DUE_WEATHER, DELAY_DUE_NAS, DELAY_DUE_SECURITY, DELAY_DUE_LATE_AIRCRAFT, YEAR, MONTH, MONTH_NAME, ROUTE, DELAYED, DELAY_CATEGORY, FLIGHT_STATUS, AIRPORT_FLIGHTS, MAJOR_AIRPORT, ROUTE_FLIGHTS, MAJOR_ROUTE]
Index: []

[0 rows x 43 columns]


In [ ]:
# Check delays with impossible times

delay_cols = [
    "DELAY_DUE_CARRIER", # Delayed flights should not have negative values
    "DELAY_DUE_WEATHER",
    "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY",
    "DELAY_DUE_LATE_AIRCRAFT"
]

for col in delay_cols:
    print(col, df[col].min())

DELAY_DUE_CARRIER 0.0
DELAY_DUE_WEATHER 0.0
DELAY_DUE_NAS 0.0
DELAY_DUE_SECURITY 0.0
DELAY_DUE_LATE_AIRCRAFT 0.0


In [ ]:
tableau_cols = [
    # Date fields
    "FL_DATE",
    "YEAR",
    "MONTH",
    "MONTH_NAME",

    # Airline fields
    "AIRLINE",
    "AIRLINE_CODE",

    # Airport fields
    "ORIGIN",
    "ORIGIN_CITY",
    "DEST",
    "DEST_CITY",

    # Route field
    "ROUTE",

    # Performance metrics
    "ARR_DELAY",
    "DEP_DELAY",
    "CANCELLED",
    "DIVERTED",
    "DELAYED",
    "DELAY_CATEGORY",
    "FLIGHT_STATUS",

    # Flight characteristics
    "DISTANCE",

    # Delay causes
    "DELAY_DUE_CARRIER",
    "DELAY_DUE_WEATHER",
    "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY",
    "DELAY_DUE_LATE_AIRCRAFT",

    # Airport filtering
    "AIRPORT_FLIGHTS",
    "MAJOR_AIRPORT",

    # Route filtering
    "ROUTE_FLIGHTS",
    "MAJOR_ROUTE"
]

# Create Tableau dataset
tableau_df = df[tableau_cols].copy()

# Check result
print(tableau_df.shape)
print(tableau_df.info())

(3000000, 28)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 28 columns):
 #   Column                   Dtype         
---  ------                   -----         
 0   FL_DATE                  datetime64[ns]
 1   YEAR                     int32         
 2   MONTH                    int32         
 3   MONTH_NAME               object        
 4   AIRLINE                  object        
 5   AIRLINE_CODE             object        
 6   ORIGIN                   object        
 7   ORIGIN_CITY              object        
 8   DEST                     object        
 9   DEST_CITY                object        
 10  ROUTE                    object        
 11  ARR_DELAY                float64       
 12  DEP_DELAY                float64       
 13  CANCELLED                float64       
 14  DIVERTED                 float64       
 15  DELAYED                  int64         
 16  DELAY_CATEGORY           object        
 17  FLIGHT_STATUS

In [ ]:
print(tableau_df.shape)

print(tableau_df["AIRLINE"].nunique())
print(tableau_df["ORIGIN"].nunique())
print(tableau_df["ROUTE"].nunique())

print(tableau_df["MAJOR_AIRPORT"].value_counts())

print(tableau_df["MAJOR_ROUTE"].value_counts())

print(tableau_df["DELAY_CATEGORY"].value_counts())

(3000000, 28)
18
380
7785
MAJOR_AIRPORT
Yes    2454981
No      545019
Name: count, dtype: int64
MAJOR_ROUTE
No     1635655
Yes    1364345
Name: count, dtype: int64
DELAY_CATEGORY
On Time / Early    1935655
Minor Delay         462858
Moderate Delay      340154
Major Delay         182193
Cancelled            79140
Name: count, dtype: int64


In [ ]:
# Create local output folder
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

# Define file paths
csv_path = os.path.join(output_dir, "usairline_dashboard_data.csv")

# Save dataset locally
tableau_df.to_csv("usairline_dashboard_data.csv",index=False)

print(f"Dataset saved to: {csv_path}")

Dataset saved to: data/usairline_dashboard_data.csv
